In [33]:
import pandas as pd
import re
from datasets import load_dataset
from bs4 import BeautifulSoup

# 1. Load a well-known LeetCode dataset from Hugging Face
# This dataset contains the actual problem descriptions (content)
print("Downloading dataset from Hugging Face...")
hf_dataset = load_dataset("greengerong/leetcode", split="train")


# 2. Convert to a Pandas DataFrame for easier manipulation
df = hf_dataset.to_pandas()
df.to_csv("../data/leetcode_raw.csv", index=False)
df.head()

,id,slug,title,difficulty,content,java,c++,python,javascript
0,1,two-sum,Two Sum,Easy,Given an array of integers `nums` and an integ...,\n ```java\nimport java.util.HashMap;\nimpo...,\n ```cpp\n#include <vector>\n#include <uno...,"\n ```python\ndef twoSum(nums, target):\n ...","\n ```javascript\nfunction twoSum(nums, tar..."
1,2,add-two-numbers,Add Two Numbers,Medium,You are given two **non-empty** linked lists r...,\n ```java\npublic class ListNode {\n in...,\n ```cpp\nstruct ListNode {\n int val;\...,\n ```python\nclass ListNode:\n def __in...,"\n ```javascript\nfunction ListNode(val, ne..."
2,3,longest-substring-without-repeating-characters,Longest Substring Without Repeating Characters,Medium,"Given a string `s`, find the length of the **l...",\n ```java\nimport java.util.HashSet;\nimpo...,\n ```cpp\n#include <string>\n#include <uno...,\n ```python\ndef length_of_longest_substri...,\n ```javascript\nfunction lengthOfLongestS...
3,4,median-of-two-sorted-arrays,Median of Two Sorted Arrays,Hard,Given two sorted arrays `nums1` and `nums2` of...,\n ```java\npublic double findMedianSortedA...,\n ```cpp\ndouble findMedianSortedArrays(ve...,\n ```python\ndef findMedianSortedArrays(nu...,\n ```javascript\nfunction findMedianSorted...
4,5,longest-palindromic-substring,Longest Palindromic Substring,Medium,"Given a string `s`, return _the longest_ _pali...",\n ```java\npublic String longestPalindromi...,\n ```cpp\n#include <string>\n\nstd::string...,\n ```python\ndef longest_palindromic_subst...,\n ```javascript\nfunction longestPalindrom...


In [20]:
clean_df = df.copy()

# 2. Fill nulls to prevent data erasure during concatenation
clean_df[['title', 'content', 'python']] = clean_df[['title', 'content', 'python']].fillna("")

# 3. Construct the semantic payload
clean_df['Searchable_Text'] = clean_df['title'] + " " + clean_df['content'] + " " + clean_df['python']

# 4. Define the rigorous text cleaning function
def clean_full_text(raw_text):
    if not isinstance(raw_text, str): return ""
    
    # Strip HTML tags
    text = BeautifulSoup(raw_text, "html.parser").get_text(separator=" ")
    
    # Standardize casing
    text = text.lower()
    
    # Remove markdown code blocks (like ```python)
    text = re.sub(r'```[a-z]*', ' ', text)
    
    # Keep alphanumeric, basic punctuation, and essential math/code operators
    text = re.sub(r'[^a-z0-9\s.,=+\-<>()\[\]]', ' ', text)
    
    # Collapse multiple spaces and trim edges
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# 5. Apply the cleaning pipeline
print("Applying HTML stripping and regex cleaning...")
clean_df['Searchable_Text'] = clean_df['Searchable_Text'].apply(clean_full_text)

# 6. Isolate the target columns for the search engine index
# We keep 'slug' as a unique identifier and 'title' for human-readable results
clean_df = clean_df[['slug', 'title', 'difficulty', 'Searchable_Text']]

# 7. Save to disk to establish a checkpoint
clean_df.to_csv("../data/leetcode_clean.csv", index=False)

print("Feature engineering complete.")
clean_df.head()

Applying HTML stripping and regex cleaning...
Feature engineering complete.


,slug,title,difficulty,Searchable_Text
0,two-sum,Two Sum,Easy,two sum given an array of integers nums and an...
1,add-two-numbers,Add Two Numbers,Medium,add two numbers you are given two non-empty li...
2,longest-substring-without-repeating-characters,Longest Substring Without Repeating Characters,Medium,longest substring without repeating characters...
3,median-of-two-sorted-arrays,Median of Two Sorted Arrays,Hard,median of two sorted arrays given two sorted a...
4,longest-palindromic-substring,Longest Palindromic Substring,Medium,longest palindromic substring given a string s...


In [21]:
# Load both datasets
# Make sure "Leetcode.csv" (Kaggle) and "leetcode_clean.csv" (Hugging Face) are in your directory
kaggle_df = pd.read_csv("../data/Leetcode.csv")
hf_df = pd.read_csv("../data/leetcode_clean.csv")

print(f"Kaggle Rows: {len(kaggle_df)} | Hugging Face Rows: {len(hf_df)}")

Kaggle Rows: 3647 | Hugging Face Rows: 2360


In [22]:
# 2. Create a reliable Primary Key for the Kaggle dataset
# The Kaggle 'Link' looks like: https://leetcode.com/problems/two-sum/
# We can extract 'two-sum' to perfectly match the 'slug' in the Hugging Face dataset
def extract_slug(url):
    if not isinstance(url, str): return ""
    # Split by '/' and grab the problem name, ignoring trailing slashes
    parts = [p for p in url.split('/') if p]
    return parts[-1] if parts else ""

print("Extracting join keys...")
kaggle_df['slug'] = kaggle_df['Link'].apply(extract_slug)

# 3. Perform an Inner Join (Merge)
# We only keep problems that exist in BOTH datasets to ensure data quality
enriched_df = pd.merge(
    hf_df, 
    kaggle_df[['ID', 'slug', 'Link', 'Topics', 'Similar Questions']], 
    on='slug', 
    how='inner'
)

# 4. Handle any newly introduced nulls from the Kaggle side
enriched_df[['Topics', 'Similar Questions']] = enriched_df[['Topics', 'Similar Questions']].fillna("None")

# 5. Rearrange columns for a clean, logical structure
final_columns = [
    'ID', 'title', 'slug', 'difficulty', 
    'Topics', 'Searchable_Text', 'Similar Questions', 'Link'
]
enriched_df = enriched_df[final_columns]

# 6. Save the Golden Record
enriched_df.to_csv("../data/leetcode_final.csv", index=False)

print(f"Merge Complete! Final enriched dataset has {len(enriched_df)} rows.")
enriched_df.head()

Extracting join keys...
Merge Complete! Final enriched dataset has 2359 rows.


,ID,title,slug,difficulty,Topics,Searchable_Text,Similar Questions,Link
0,1,Two Sum,two-sum,Easy,"Array, Hash Table",two sum given an array of integers nums and an...,"[{""title"": ""3Sum"", ""titleSlug"": ""3sum"", ""diffi...",https://leetcode.com/problems/two-sum/
1,2,Add Two Numbers,add-two-numbers,Medium,"Linked List, Math, Recursion",add two numbers you are given two non-empty li...,"[{""title"": ""Multiply Strings"", ""titleSlug"": ""m...",https://leetcode.com/problems/add-two-numbers/
2,3,Longest Substring Without Repeating Characters,longest-substring-without-repeating-characters,Medium,"Hash Table, String, Sliding Window",longest substring without repeating characters...,"[{""title"": ""Longest Substring with At Most Two...",https://leetcode.com/problems/longest-substrin...
3,4,Median of Two Sorted Arrays,median-of-two-sorted-arrays,Hard,"Array, Binary Search, Divide and Conquer",median of two sorted arrays given two sorted a...,"[{""title"": ""Median of a Row Wise Sorted Matrix...",https://leetcode.com/problems/median-of-two-so...
4,5,Longest Palindromic Substring,longest-palindromic-substring,Medium,"Two Pointers, String, Dynamic Programming",longest palindromic substring given a string s...,"[{""title"": ""Shortest Palindrome"", ""titleSlug"":...",https://leetcode.com/problems/longest-palindro...


In [26]:
import re

def extract_code_skeleton(code_str):
    """
    Extracts function headers and primary data structure initializations.
    Discards the logic body to minimize token noise.
    """
    if not isinstance(code_str, str) or code_str.strip() == "":
        return ""
    
    lines = code_str.split('\n')
    skeleton = []
    found_header = False
    capture_limit = 5 # Number of lines to scan for initializations after header
    captured_count = 0

    for line in lines:
        clean_line = line.strip()
        
        # Skip comments and empty lines
        if not clean_line or clean_line.startswith(('#', '//', '/*')):
            continue
            
        # 1. Capture the Function/Class Header
        if any(keyword in clean_line for keyword in ['def ', 'class ', 'public ', 'vector<']):
            if ':' in clean_line or '{' in clean_line or ')' in clean_line:
                skeleton.append(clean_line)
                found_header = True
                continue
        
        # 2. Capture Initializations (Semantic Anchors)
        if found_header and captured_count < capture_limit:
            # We look for assignments like dp =, q =, seen = set()
            if '=' in clean_line and not any(k in clean_line for k in ['for ', 'if ', 'while ']):
                skeleton.append(clean_line)
            
            captured_count += 1
            
        # Stop once we've looked far enough or hit the body
        if captured_count >= capture_limit:
            break
            
    return " ".join(skeleton)

# Apply to your Golden Record DataFrame
print("Pruning code to extract structural skeletons...")
# We apply this to the raw 'python' column BEFORE the final text cleaning
enriched_df['Code_Skeleton'] = df['python'].apply(extract_code_skeleton)

# Update Searchable_Text to use the Skeleton instead of the full code
enriched_df['Searchable_Text'] = (
    enriched_df['title'] + " " + 
    df['content'] + " " + 
    enriched_df['Code_Skeleton']
)

# Run the standard cleaning on the final string
# (Using the clean_full_text function we defined earlier)
enriched_df['Searchable_Text'] = enriched_df['Searchable_Text'].apply(clean_full_text)

print("Code Skeleton extraction complete.")
enriched_df[['title', 'Code_Skeleton']].head()

Pruning code to extract structural skeletons...
Code Skeleton extraction complete.


,title,Code_Skeleton
0,Two Sum,"def twoSum(nums, target): map = {} complement ..."
1,Add Two Numbers,"class ListNode: def __init__(self, val=0, next..."
2,Longest Substring Without Repeating Characters,def length_of_longest_substring(s: str) -> int...
3,Median of Two Sorted Arrays,"def findMedianSortedArrays(nums1, nums2): x, y..."
4,Longest Palindromic Substring,def longest_palindromic_substring(s: str) -> s...


In [29]:
enriched_df.head()

,ID,title,slug,difficulty,Topics,Searchable_Text,Similar Questions,Link,Code_Skeleton,Clean_Similar
0,1,Two Sum,two-sum,Easy,"Array, Hash Table","two sum two sum array, hash table given an arr...","[{""title"": ""3Sum"", ""titleSlug"": ""3sum"", ""diffi...",https://leetcode.com/problems/two-sum/,"def twoSum(nums, target): map = {} complement ...",3Sum 4Sum Two Sum II - Input Array Is Sorted T...
1,2,Add Two Numbers,add-two-numbers,Medium,"Linked List, Math, Recursion","add two numbers add two numbers linked list, m...","[{""title"": ""Multiply Strings"", ""titleSlug"": ""m...",https://leetcode.com/problems/add-two-numbers/,"class ListNode: def __init__(self, val=0, next...",Multiply Strings Add Binary Sum of Two Integer...
2,3,Longest Substring Without Repeating Characters,longest-substring-without-repeating-characters,Medium,"Hash Table, String, Sliding Window",longest substring without repeating characters...,"[{""title"": ""Longest Substring with At Most Two...",https://leetcode.com/problems/longest-substrin...,def length_of_longest_substring(s: str) -> int...,Longest Substring with At Most Two Distinct Ch...
3,4,Median of Two Sorted Arrays,median-of-two-sorted-arrays,Hard,"Array, Binary Search, Divide and Conquer",median of two sorted arrays median of two sort...,"[{""title"": ""Median of a Row Wise Sorted Matrix...",https://leetcode.com/problems/median-of-two-so...,"def findMedianSortedArrays(nums1, nums2): x, y...",Median of a Row Wise Sorted Matrix
4,5,Longest Palindromic Substring,longest-palindromic-substring,Medium,"Two Pointers, String, Dynamic Programming",longest palindromic substring longest palindro...,"[{""title"": ""Shortest Palindrome"", ""titleSlug"":...",https://leetcode.com/problems/longest-palindro...,def longest_palindromic_substring(s: str) -> s...,Shortest Palindrome Palindrome Permutation Pal...


In [28]:
import json

def extract_similar_titles(json_str):
    """
    Extracts only the titles from the Similar Questions JSON string.
    """
    if not isinstance(json_str, str) or json_str == "None" or json_str.strip() == "":
        return ""
    try:
        # Some datasets store this as a string representation of a list of dicts
        data = json.loads(json_str.replace("'", '"')) 
        titles = [item['title'] for item in data if 'title' in item]
        return " ".join(titles)
    except:
        return ""

# 1. Extract clean metadata
print("Cleaning Similar Questions titles...")
enriched_df['Clean_Similar'] = enriched_df['Similar Questions'].apply(extract_similar_titles)

# 2. Re-construct the Final Searchable Payload
# Logic: Title (x2 weight) + Topics + Content + Code Skeleton + Similar Titles
# Note: We repeat the 'title' twice to give it natural 'Lexical Weighting' 
enriched_df['Searchable_Text'] = (
    enriched_df['title'] + " " + 
    enriched_df['title'] + " " + 
    enriched_df['Topics'] + " " + 
    df['content'] + " " + 
    enriched_df['Code_Skeleton'] + " " + 
    enriched_df['Clean_Similar']
)

# 3. Final standard cleaning
enriched_df['Searchable_Text'] = enriched_df['Searchable_Text'].apply(clean_full_text)

print("Enriched Searchable_Text constructed with Topics and Similar Questions.")

Cleaning Similar Questions titles...
Enriched Searchable_Text constructed with Topics and Similar Questions.


In [31]:
df.isnull().sum()
enriched_df['difficulty'] = enriched_df['difficulty'].fillna("Unknown")
enriched_df.to_csv("../data/leetcode_final.csv", index=False)

In [49]:
df = pd.read_csv("../data/leetcode_final.csv")
data = pd.read_csv("../data/leetcode_raw.csv")


def optimize_searchable_text(row):
    # 1. Clean the Title
    title = str(row['title']).lower()
    
    # 2. Clean the Description (Remove Markdown and limit length)
    # This keeps the 'Intent' without the 'HTML Noise'
    content = str(row['content'])
    content = re.sub(r'[*_`#]', '', content) # Strip Markdown
    content = " ".join(content.split()[:100]) # Keep only the first 100 words
    
    # 3. Prune Similar Questions
    # We only want the Top 3 most relevant ones to avoid dilution
    similar = " ".join(str(row['Clean_Similar']).split()[:5])
    
    # 4. Assembly (Title is most important -> Topics -> Code -> Snippet)
    # We triple the title for the Lexical 'Boost'
    final_text = (
        f"{title} {title} {title} " 
        f"{row['Topics']} "
        f"{row['Code_Skeleton']} "
        f"{content} "
        f"{similar}"
    )
    
    return final_text.lower().strip()

df['Searchable_Text'] = df.apply(optimize_searchable_text, axis=1)
df = df.drop(columns='Searchable_Text_Optimized')
df.head()

,ID,title,slug,difficulty,Topics,Searchable_Text,Similar Questions,Link,Code_Skeleton,Clean_Similar,content
0,1,Two Sum,two-sum,Easy,"Array, Hash Table","two sum two sum two sum array, hash table def ...","[{""title"": ""3Sum"", ""titleSlug"": ""3sum"", ""diffi...",https://leetcode.com/problems/two-sum/,"def twoSum(nums, target): map = {} complement ...",3Sum 4Sum Two Sum II - Input Array Is Sorted T...,Given an array of integers `nums` and an integ...
1,2,Add Two Numbers,add-two-numbers,Medium,"Linked List, Math, Recursion",add two numbers add two numbers add two number...,"[{""title"": ""Multiply Strings"", ""titleSlug"": ""m...",https://leetcode.com/problems/add-two-numbers/,"class ListNode: def __init__(self, val=0, next...",Multiply Strings Add Binary Sum of Two Integer...,You are given two **non-empty** linked lists r...
2,3,Longest Substring Without Repeating Characters,longest-substring-without-repeating-characters,Medium,"Hash Table, String, Sliding Window",longest substring without repeating characters...,"[{""title"": ""Longest Substring with At Most Two...",https://leetcode.com/problems/longest-substrin...,def length_of_longest_substring(s: str) -> int...,Longest Substring with At Most Two Distinct Ch...,"Given a string `s`, find the length of the **l..."
3,4,Median of Two Sorted Arrays,median-of-two-sorted-arrays,Hard,"Array, Binary Search, Divide and Conquer",median of two sorted arrays median of two sort...,"[{""title"": ""Median of a Row Wise Sorted Matrix...",https://leetcode.com/problems/median-of-two-so...,"def findMedianSortedArrays(nums1, nums2): x, y...",Median of a Row Wise Sorted Matrix,Given two sorted arrays `nums1` and `nums2` of...
4,5,Longest Palindromic Substring,longest-palindromic-substring,Medium,"Two Pointers, String, Dynamic Programming",longest palindromic substring longest palindro...,"[{""title"": ""Shortest Palindrome"", ""titleSlug"":...",https://leetcode.com/problems/longest-palindro...,def longest_palindromic_substring(s: str) -> s...,Shortest Palindrome Palindrome Permutation Pal...,"Given a string `s`, return _the longest_ _pali..."


In [50]:
df.to_csv("../data/leetcode_final.csv", index=False)
df.head()

,ID,title,slug,difficulty,Topics,Searchable_Text,Similar Questions,Link,Code_Skeleton,Clean_Similar,content
0,1,Two Sum,two-sum,Easy,"Array, Hash Table","two sum two sum two sum array, hash table def ...","[{""title"": ""3Sum"", ""titleSlug"": ""3sum"", ""diffi...",https://leetcode.com/problems/two-sum/,"def twoSum(nums, target): map = {} complement ...",3Sum 4Sum Two Sum II - Input Array Is Sorted T...,Given an array of integers `nums` and an integ...
1,2,Add Two Numbers,add-two-numbers,Medium,"Linked List, Math, Recursion",add two numbers add two numbers add two number...,"[{""title"": ""Multiply Strings"", ""titleSlug"": ""m...",https://leetcode.com/problems/add-two-numbers/,"class ListNode: def __init__(self, val=0, next...",Multiply Strings Add Binary Sum of Two Integer...,You are given two **non-empty** linked lists r...
2,3,Longest Substring Without Repeating Characters,longest-substring-without-repeating-characters,Medium,"Hash Table, String, Sliding Window",longest substring without repeating characters...,"[{""title"": ""Longest Substring with At Most Two...",https://leetcode.com/problems/longest-substrin...,def length_of_longest_substring(s: str) -> int...,Longest Substring with At Most Two Distinct Ch...,"Given a string `s`, find the length of the **l..."
3,4,Median of Two Sorted Arrays,median-of-two-sorted-arrays,Hard,"Array, Binary Search, Divide and Conquer",median of two sorted arrays median of two sort...,"[{""title"": ""Median of a Row Wise Sorted Matrix...",https://leetcode.com/problems/median-of-two-so...,"def findMedianSortedArrays(nums1, nums2): x, y...",Median of a Row Wise Sorted Matrix,Given two sorted arrays `nums1` and `nums2` of...
4,5,Longest Palindromic Substring,longest-palindromic-substring,Medium,"Two Pointers, String, Dynamic Programming",longest palindromic substring longest palindro...,"[{""title"": ""Shortest Palindrome"", ""titleSlug"":...",https://leetcode.com/problems/longest-palindro...,def longest_palindromic_substring(s: str) -> s...,Shortest Palindrome Palindrome Permutation Pal...,"Given a string `s`, return _the longest_ _pali..."
